In [19]:
import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

In [2]:
df = pd.read_csv("../sample_dataset copy.csv")

In [3]:
text_col = df.select_dtypes(include=['object']).columns[0]
texts = df[text_col].astype(str).tolist()

In [22]:
clusterer = HDBSCAN(prediction_data=True)
vectorizer = CountVectorizer(stop_words="english")
umap_model = UMAP(metric='cosine', random_state=8)
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)
embedding_model = 'all-MiniLM-L6-v2'

model = BERTopic(
    min_topic_size=10,
    embedding_model=embedding_model,
    calculate_probabilities=True,
    umap_model=umap_model,
    hdbscan_model=clusterer,
    vectorizer_model=vectorizer,
    ctfidf_model=ctfidf_model
)

# Fit the model
topics, probs = model.fit_transform(texts)

In [23]:
print(len(model.get_topics().keys()))
print(range(len(model.get_topics().keys())))

22
range(0, 22)


In [24]:
topics_list = []
# Topic -1 is outliers, we usually exclude it from the top topics list
for topic_id in range(len(model.get_topics().keys())-1):
    if topic_id == -1: continue
    topic_data = model.get_topic(topic_id)
    # Convert [(word, weight), ...] to {word: weight, ...}
    topics_list.append({word: weight for word, weight in topic_data})

In [25]:
# 2. Extract Doc-Topic assignments
# If calculate_probabilities=True, probs is a numpy array [n_docs, n_topics]
doc_topics = []
# if probs is not None:
if isinstance(probs, np.ndarray):
    # Map each doc to a dict of {topic_id: probability}
    # BERTopic's probs array corresponds to the topics in model.get_topics()
    # Topic -1 is not usually in the probs array
    topic_names = list(model.get_topics().keys())
    # Filter out -1 for probability mapping
    filtered_topics = [t for t in topic_names if t != -1]

    for doc_prob in probs:
        # Map prob index to actual topic_id
        doc_map = {filtered_topics[i]: prob for i, prob in enumerate(doc_prob)}
        doc_topics.append(doc_map)
else:
    # Fallback to hard assignment
    for t in topics:
        doc_topics.append({t: 1.0})

In [5]:
probs

array([1.        , 0.89298062, 1.        , 1.        , 0.83124065,
       1.        , 1.        , 1.        , 0.        , 0.        ,
       0.81613267, 1.        , 1.        , 0.86717782, 1.        ,
       1.        , 1.        , 0.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 0.39403541,
       0.76960107, 0.        , 1.        , 0.99510235, 1.        ,
       0.86810233, 1.        , 1.        , 1.        , 0.88651044,
       1.        , 1.        , 1.        , 1.        , 0.91612317,
       0.47696027, 0.73814699, 0.        , 1.        , 1.        ,
       0.        , 0.        , 1.        , 0.95161707, 0.        ,
       1.        , 1.        , 0.77710903, 1.        , 1.        ,
       0.99167602, 1.        , 1.        , 0.90147729, 1.        ,
       0.        , 0.86087095, 1.        , 1.        , 0.74438814,
       0.        , 1.        , 1.        , 1.        , 0.90458

In [6]:
topic_info = model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,82,-1_waste_species_accuracy_proposed,"[waste, species, accuracy, proposed, smart, le...",[Highlights: What are the main findings? For s...
1,0,442,0_algorithmic_administration_sector_making,"[algorithmic, administration, sector, making, ...",[Introduction: The widespread adoption of info...
2,1,33,1_budget_neural_project_border,"[budget, neural, project, border, projects, li...",[Government bodies around the world are going ...
3,2,28,2_air_buildings_pollution_deep,"[air, buildings, pollution, deep, fog, learnin...",[Highlights: What are the main findings? CO po...
4,3,25,3_covid_19_health_epidemic,"[covid, 19, health, epidemic, chest, pandemic,...",[The worldwide pandemic Coronavirus (Covid-19)...
5,4,22,4_traffic_congestion_parking_transport,"[traffic, congestion, parking, transport, rfr,...",[Finding an available parking place has been c...
6,5,21,5_problem_scheduling_genetic_facility,"[problem, scheduling, genetic, facility, algor...",[The environment in earthquake-stricken areas ...
7,6,20,6_green_environmental_japanese_energy,"[green, environmental, japanese, energy, city,...",[Smart cities play a vital role in the growth ...
8,7,20,7_sentiment_opinion_tweets_text,"[sentiment, opinion, tweets, text, arabic, twi...",[Social media materialized as an influential p...
9,8,14,8_water_land_crop_urban,"[water, land, crop, urban, safety, mg, bpnn, s...",[Cropping pattern planning is important to avo...


In [7]:
model.visualize_topics(use_ctfidf=True, custom_labels=True)

In [8]:
hierarchical_topic = model.hierarchical_topics(df["Abstract"])
model.visualize_hierarchy(hierarchical_topics=hierarchical_topic)

100%|██████████| 20/20 [00:00<00:00, 725.34it/s]
